In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.1 MB/s eta 0:00:00


In [ ]:
import os
import sys
import time
import random
import json
import gc
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import pennylane as qml
from pennylane import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from tqdm.auto import tqdm

class Config:
    MODE = 'FULL' 
    SEED = 42
    GRID_H = 2
    GRID_W = 2
    N_WIRES = GRID_H * GRID_W
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    GLOBAL_RESULTS = {}
    DATA_ROOT = './data'
    OUTPUT_BASE = '/kaggle/working/outputs' 
    RUN_DIR = None
    LOGGER = None

config = Config()

def seed_everything(random_seed=42):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def create_run_dir():
    current_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_folder_name = f"{current_timestamp}_Phase4_5Fold_Inference"
    config.RUN_DIR = os.path.join(config.OUTPUT_BASE, run_folder_name)
    if not os.path.exists(config.RUN_DIR):
        os.makedirs(config.RUN_DIR)

def setup_logger():
    log_file_path = os.path.join(config.RUN_DIR, "execution.log")
    logger = logging.getLogger("Experiment")
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    
    file_handler = logging.FileHandler(log_file_path)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    
    config.LOGGER = logger

def save_to_json(data_dict, filename="final_results"):
    def json_serializer(obj):
        if isinstance(obj, (np.int64, np.int32)): return int(obj)
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, torch.Tensor): return obj.item()
        if isinstance(obj, np.ndarray): return obj.tolist()
        return obj
    
    final_filename = f"{filename}_{config.MODE}.json"
    save_path = os.path.join(config.RUN_DIR, final_filename)
    with open(save_path, 'w') as f:
        json.dump(data_dict, f, default=json_serializer, indent=4)

def generate_grid_topology(height, width):
    h_pairs, v_pairs = [], []
    for row in range(height):
        for col in range(width - 1):
            h_pairs.append([row * width + col, row * width + col + 1])
    for row in range(height - 1):
        for col in range(width):
            v_pairs.append([row * width + col, (row + 1) * width + col])
    return h_pairs, v_pairs

def get_test_dataloader(dataset_name, data_size, batch_size, fold_seed):
    transform_pipeline = transforms.Compose([
        transforms.Resize((14, 14)),
        transforms.ToTensor()
    ])
    dataset_class = getattr(datasets, dataset_name)
    test_full = dataset_class(root=config.DATA_ROOT, train=False, download=True, transform=transform_pipeline)

    total_test = len(test_full)
    subset_size = min(data_size, total_test)
    rng = torch.Generator().manual_seed(fold_seed)
    indices = torch.randperm(total_test, generator=rng)[:subset_size]
    test_subset = Subset(test_full, indices)
    
    kwargs = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}
    return DataLoader(test_subset, batch_size=batch_size, shuffle=False, **kwargs)

class LPQCNN(nn.Module):
    def __init__(self, n_filters, time_dyn, time_steps, n_classes=10, kernel_size=2, stride=1, noise_prob=0.0):
        super().__init__()
        self.n_filters = n_filters
        self.time_dyn = time_dyn
        self.time_steps = time_steps
        self.n_wires = config.N_WIRES
        self.h_pairs, self.v_pairs = generate_grid_topology(config.GRID_H, config.GRID_W)
        self.hamiltonian_params = nn.Parameter(torch.randn(n_filters, 6))
        self.noise_prob = noise_prob
        
        if self.noise_prob > 0.0:
            self.dev = qml.device("default.mixed", wires=self.n_wires)
        else:
            self.dev = qml.device("default.qubit", wires=self.n_wires)
        
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def circuit(inputs, params):
            qml.AngleEmbedding(inputs * np.pi, wires=range(self.n_wires), rotation='Y')
            dt = self.time_dyn / self.time_steps
            jx_h, jy_h, jz_h, jx_v, jy_v, jz_v = params
            
            for _ in range(self.time_steps):
                for w in self.h_pairs:
                    qml.IsingXX(2*jx_h*dt, wires=w); qml.IsingYY(2*jy_h*dt, wires=w); qml.IsingZZ(2*jz_h*dt, wires=w)
                for w in self.v_pairs:
                    qml.IsingXX(2*jx_v*dt, wires=w); qml.IsingYY(2*jy_v*dt, wires=w); qml.IsingZZ(2*jz_v*dt, wires=w)
                
                if self.noise_prob > 0.0:
                    for w in range(self.n_wires):
                        qml.DepolarizingChannel(self.noise_prob, wires=w)
            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_wires)]
        
        self.qnode = circuit
        dim = 14
        out_dim = (dim - kernel_size) // stride + 1
        self.fc = nn.Linear(n_filters * self.n_wires * out_dim * out_dim, n_classes)

    def forward(self, x):
        batch_size, _, _, _ = x.shape
        patches = F.unfold(x, kernel_size=2, stride=1) 
        num_patches = patches.shape[-1]
        x_flat = patches.transpose(1, 2).reshape(-1, self.n_wires) * 0.5
        x_flat = x_flat.to(config.DEVICE)
        
        outputs = []
        for k in range(self.n_filters):
            res = self.qnode(x_flat, self.hamiltonian_params[k])
            outputs.append(torch.stack(res, dim=-1) if isinstance(res, (list, tuple)) else res)
            
        out_tensor = torch.stack(outputs, dim=1).view(batch_size, -1)
        return self.fc(out_tensor.float())

def run_phase_4_robustness(best_configs, pretrained_paths, n_folds=5, data_size=5000):
    config.LOGGER.info("Starting Phase 4: 5-Fold Noise Resilience Evaluation")
    noise_levels = [0, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2]
    
    for ds_name, base_cfg in best_configs.items():
        config.LOGGER.info(f"\nEvaluating Dataset: {ds_name}")
        fold_accuracies = np.zeros((n_folds, len(noise_levels)))
        
        for fold in range(n_folds):
            fold_seed = config.SEED + fold
            config.LOGGER.info(f"--- Fold {fold+1}/{n_folds} (Seed: {fold_seed}) ---")
            
            weights = torch.load(pretrained_paths[ds_name], map_location=config.DEVICE, weights_only=False)
            if isinstance(weights, nn.Module): weights = weights.state_dict()

            for i, noise in enumerate(noise_levels):
                loader = get_test_dataloader(ds_name, data_size, 128, fold_seed)
                model = LPQCNN(
                    n_filters=base_cfg['n_filters'], 
                    time_dyn=base_cfg['time'], 
                    time_steps=base_cfg['time_steps'], 
                    noise_prob=noise
                ).to(config.DEVICE)
                model.load_state_dict(weights)
                model.eval()
                
                correct, total = 0, 0
                with torch.no_grad():
                    for inputs, targets in loader:
                        inputs, targets = inputs.to(config.DEVICE), targets.to(config.DEVICE)
                        outputs = model(inputs)
                        correct += (outputs.argmax(dim=1) == targets).sum().item()
                        total += inputs.size(0)
                
                acc = correct / total
                fold_accuracies[fold, i] = acc
                config.LOGGER.info(f"Noise {noise*100}%: Accuracy = {acc:.4f}")
                
                del model; gc.collect(); torch.cuda.empty_cache()

        mean_accs = np.mean(fold_accuracies, axis=0)
        std_accs = np.std(fold_accuracies, axis=0)
        
        plt.figure(figsize=(7, 5))
        plt.plot(noise_levels, mean_accs, marker='s', color='#1f77b4', label='Mean Accuracy')
        plt.fill_between(noise_levels, mean_accs - std_accs, mean_accs + std_accs, alpha=0.2, color='#1f77b4', label='±1 Std Dev')
        
        plt.xlabel("Depolarizing Noise Probability ($p$)")
        plt.ylabel("Test Accuracy")
        plt.xticks(noise_levels, [f"{int(n*100)}%" for n in noise_levels])
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        
        pdf_path = os.path.join(config.RUN_DIR, f"Robustness_{ds_name}.pdf")
        plt.savefig(pdf_path, format='pdf', bbox_inches='tight')
        plt.close()
        
        config.GLOBAL_RESULTS[ds_name] = {
            "noise_levels": noise_levels,
            "mean": mean_accs.tolist(),
            "std": std_accs.tolist(),
            "raw_folds": fold_accuracies.tolist()
        }

if __name__ == "__main__":
    create_run_dir()
    setup_logger()
    seed_everything(config.SEED)
    
    PRETRAINED_PATHS = {
        'MNIST': '/kaggle/input/datasets/luongkhaichuong/lpqcnn-param/model_P1_MNIST_FULL.pth',
        'FashionMNIST': '/kaggle/input/datasets/luongkhaichuong/lpqcnn-param/model_P1_FashionMNIST_FULL.pth'
    }
    
    BEST_CONFIGS = {
        'MNIST': {'time': 2.0, 'time_steps': 4, 'n_filters': 3},
        'FashionMNIST': {'time': 5.0, 'time_steps': 5, 'n_filters': 6}
    }
    
    run_phase_4_robustness(BEST_CONFIGS, PRETRAINED_PATHS, n_folds=5, data_size=5000)
    save_to_json(config.GLOBAL_RESULTS, "phase4_final_metrics")

2026-04-15 20:24:19 | INFO | Starting Phase 4: 5-Fold Noise Resilience Evaluation
2026-04-15 20:24:19 | INFO | 
Evaluating Dataset: MNIST
2026-04-15 20:24:19 | INFO | --- Fold 1/5 (Seed: 42) ---


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.67MB/s]


2026-04-15 20:24:28 | INFO | Noise 0%: Accuracy = 0.9806
2026-04-15 20:27:48 | INFO | Noise 1.0%: Accuracy = 0.9786
2026-04-15 20:31:09 | INFO | Noise 3.0%: Accuracy = 0.9582
2026-04-15 20:34:30 | INFO | Noise 5.0%: Accuracy = 0.9198
2026-04-15 20:37:50 | INFO | Noise 10.0%: Accuracy = 0.7408
2026-04-15 20:41:11 | INFO | Noise 15.0%: Accuracy = 0.5078
2026-04-15 20:44:34 | INFO | Noise 20.0%: Accuracy = 0.3456
2026-04-15 20:44:34 | INFO | --- Fold 2/5 (Seed: 43) ---
2026-04-15 20:44:38 | INFO | Noise 0%: Accuracy = 0.9794
2026-04-15 20:47:58 | INFO | Noise 1.0%: Accuracy = 0.9762
2026-04-15 20:51:19 | INFO | Noise 3.0%: Accuracy = 0.9562
2026-04-15 20:54:39 | INFO | Noise 5.0%: Accuracy = 0.9154
2026-04-15 20:58:01 | INFO | Noise 10.0%: Accuracy = 0.7258
2026-04-15 21:01:23 | INFO | Noise 15.0%: Accuracy = 0.5022
2026-04-15 21:04:44 | INFO | Noise 20.0%: Accuracy = 0.3446
2026-04-15 21:04:44 | INFO | --- Fold 3/5 (Seed: 44) ---
2026-04-15 21:04:49 | INFO | Noise 0%: Accuracy = 0.9772
2

100%|██████████| 26.4M/26.4M [00:02<00:00, 13.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 209kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.89MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.1MB/s]


2026-04-15 22:05:19 | INFO | Noise 0%: Accuracy = 0.8682
2026-04-15 22:12:31 | INFO | Noise 1.0%: Accuracy = 0.8692
2026-04-15 22:19:57 | INFO | Noise 3.0%: Accuracy = 0.8462
2026-04-15 22:27:25 | INFO | Noise 5.0%: Accuracy = 0.8018
2026-04-15 22:34:52 | INFO | Noise 10.0%: Accuracy = 0.6592
2026-04-15 22:42:17 | INFO | Noise 15.0%: Accuracy = 0.5288
2026-04-15 22:49:43 | INFO | Noise 20.0%: Accuracy = 0.3164
2026-04-15 22:49:43 | INFO | --- Fold 2/5 (Seed: 43) ---
2026-04-15 22:49:53 | INFO | Noise 0%: Accuracy = 0.8646
2026-04-15 22:57:15 | INFO | Noise 1.0%: Accuracy = 0.8626
2026-04-15 23:04:31 | INFO | Noise 3.0%: Accuracy = 0.8402
2026-04-15 23:11:44 | INFO | Noise 5.0%: Accuracy = 0.7994
2026-04-15 23:19:07 | INFO | Noise 10.0%: Accuracy = 0.6604
2026-04-15 23:26:33 | INFO | Noise 15.0%: Accuracy = 0.5320
2026-04-15 23:33:58 | INFO | Noise 20.0%: Accuracy = 0.3132
2026-04-15 23:33:59 | INFO | --- Fold 3/5 (Seed: 44) ---
2026-04-15 23:34:09 | INFO | Noise 0%: Accuracy = 0.8682
2